In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.9 MB/s  0:00:00m0:00:0100:01

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:

import numpy as np
import pandas as pd


# --------------------------------------------------
# 10. Load Iris dataset and select four features
# --------------------------------------------------

df = pd.read_csv("Iris.csv")  # Download Iris.csv from Kaggle

# Kaggle dataset may contain an Id column; ignore it.
feature_names = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]

X = df[feature_names].values.astype(float)

print("Original feature matrix shape:", X.shape)


# --------------------------------------------------
# 11. Standardize the feature matrix
# --------------------------------------------------

def standardize(X):
    rows, cols = X.shape
    Z = np.zeros((rows, cols))

    for j in range(cols):
        total = 0.0

        # Calculate mean
        for i in range(rows):
            total += X[i, j]

        mean = total / rows

        # Calculate population standard deviation
        squared_sum = 0.0
        for i in range(rows):
            squared_sum += (X[i, j] - mean) ** 2

        std = (squared_sum / rows) ** 0.5

        # Standardize each value
        for i in range(rows):
            if std != 0:
                Z[i, j] = (X[i, j] - mean) / std
            else:
                Z[i, j] = 0.0

    return Z


Z = standardize(X)
print("\nStandardized data (first 5 rows):")
print(Z[:5])


# --------------------------------------------------
# 12. Calculate covariance matrix C
# --------------------------------------------------

def covariance_matrix(Z):
    n, p = Z.shape
    C = np.zeros((p, p))

    for i in range(p):
        for j in range(p):
            total = 0.0

            for k in range(n):
                total += Z[k, i] * Z[k, j]

            # Sample covariance: divide by n - 1
            C[i, j] = total / (n - 1)

    return C


C = covariance_matrix(Z)

print("\nCovariance matrix C:")
print(C)


# --------------------------------------------------
# 13. Jacobi method for eigenvalue decomposition
# --------------------------------------------------

def jacobi_eigen(A, tolerance=1e-10, max_iterations=1000):
    n = len(A)

    # Work on a copy; V stores eigenvectors
    B = A.astype(float).copy()
    V = np.eye(n)

    for iteration in range(max_iterations):

        # Find largest off-diagonal element
        largest = 0.0
        p, q = 0, 1

        for i in range(n):
            for j in range(i + 1, n):
                if abs(B[i, j]) > largest:
                    largest = abs(B[i, j])
                    p, q = i, j

        # Stop when off-diagonal elements are sufficiently small
        if largest < tolerance:
            break

        # Calculate Jacobi rotation
        tau = (B[q, q] - B[p, p]) / (2.0 * B[p, q])

        if tau >= 0:
            t = 1.0 / (tau + (1.0 + tau * tau) ** 0.5)
        else:
            t = -1.0 / (-tau + (1.0 + tau * tau) ** 0.5)

        c = 1.0 / (1.0 + t * t) ** 0.5
        s = t * c

        # Save old values before updating
        app = B[p, p]
        aqq = B[q, q]
        apq = B[p, q]

        # Update diagonal elements
        B[p, p] = app - t * apq
        B[q, q] = aqq + t * apq

        # Make selected off-diagonal elements zero
        B[p, q] = 0.0
        B[q, p] = 0.0

        # Update remaining elements of B
        for r in range(n):
            if r != p and r != q:
                arp = B[r, p]
                arq = B[r, q]

                B[r, p] = c * arp - s * arq
                B[p, r] = B[r, p]

                B[r, q] = s * arp + c * arq
                B[q, r] = B[r, q]

        # Update eigenvector matrix V
        for r in range(n):
            vrp = V[r, p]
            vrq = V[r, q]

            V[r, p] = c * vrp - s * vrq
            V[r, q] = s * vrp + c * vrq

    # Eigenvalues are now approximately on the diagonal
    eigenvalues = np.zeros(n)

    for i in range(n):
        eigenvalues[i] = B[i, i]

    return eigenvalues, V


eigenvalues, eigenvectors = jacobi_eigen(C)


# --------------------------------------------------
# 14. Sort eigenvalues in descending order
# --------------------------------------------------

# Manual sorting: keep eigenvectors matched to eigenvalues
for i in range(len(eigenvalues)):
    max_index = i

    for j in range(i + 1, len(eigenvalues)):
        if eigenvalues[j] > eigenvalues[max_index]:
            max_index = j

    # Swap eigenvalues
    temp = eigenvalues[i]
    eigenvalues[i] = eigenvalues[max_index]
    eigenvalues[max_index] = temp

    # Swap corresponding eigenvector columns
    for r in range(eigenvectors.shape[0]):
        temp = eigenvectors[r, i]
        eigenvectors[r, i] = eigenvectors[r, max_index]
        eigenvectors[r, max_index] = temp


print("\nSorted eigenvalues:")
print(eigenvalues)

print("\nCorresponding eigenvectors (columns):")
print(eigenvectors)


# --------------------------------------------------
# 15. Explained variance and cumulative variance
# --------------------------------------------------

total_variance = 0.0

for i in range(len(eigenvalues)):
    total_variance += eigenvalues[i]

explained_variance = np.zeros(len(eigenvalues))
cumulative_variance = np.zeros(len(eigenvalues))

running_total = 0.0

for i in range(len(eigenvalues)):
    explained_variance[i] = (
        eigenvalues[i] / total_variance
    ) * 100.0

    running_total += explained_variance[i]
    cumulative_variance[i] = running_total


print("\nExplained variance (%):")
print(explained_variance)

print("\nCumulative explained variance (%):")
print(cumulative_variance)


# --------------------------------------------------
# 16. Principal direction
# --------------------------------------------------

# First eigenvector corresponds to the largest eigenvalue
principal_direction = eigenvectors[:, 0]

print("\nLargest eigenvalue:")
print(eigenvalues[0])

print("\nPrincipal direction (first eigenvector):")
print(principal_direction)


# --------------------------------------------------
# 17. Reconstruct covariance matrix
# --------------------------------------------------

# C = V * Lambda * V.T
# Construct V * Lambda manually by scaling columns of V

V_lambda = np.zeros(eigenvectors.shape)

for i in range(len(eigenvalues)):
    for j in range(eigenvectors.shape[0]):
        V_lambda[j, i] = eigenvectors[j, i] * eigenvalues[i]

C_reconstructed = V_lambda @ eigenvectors.T

print("\nReconstructed covariance matrix:")
print(C_reconstructed)

# Calculate reconstruction error manually
error_squared = 0.0

for i in range(C.shape[0]):
    for j in range(C.shape[1]):
        difference = C[i, j] - C_reconstructed[i, j]
        error_squared += difference * difference

reconstruction_error = error_squared ** 0.5

print("\nReconstruction error (Frobenius norm):")
print(reconstruction_error)


Original feature matrix shape: (150, 4)

Standardized data (first 5 rows):
[[-0.90068117  1.03205722 -1.3412724  -1.31297673]
 [-1.14301691 -0.1249576  -1.3412724  -1.31297673]
 [-1.38535265  0.33784833 -1.39813811 -1.31297673]
 [-1.50652052  0.10644536 -1.2844067  -1.31297673]
 [-1.02184904  1.26346019 -1.3412724  -1.31297673]]

Covariance matrix C:
[[ 1.00671141 -0.11010327  0.87760486  0.82344326]
 [-0.11010327  1.00671141 -0.42333835 -0.358937  ]
 [ 0.87760486 -0.42333835  1.00671141  0.96921855]
 [ 0.82344326 -0.358937    0.96921855  1.00671141]]

Sorted eigenvalues:
[2.93035378 0.92740362 0.14834223 0.02074601]

Corresponding eigenvectors (columns):
[[ 0.52237162  0.37231836  0.72101681  0.26199559]
 [-0.26335492  0.92555649 -0.24203288 -0.12413481]
 [ 0.58125401  0.02109478 -0.14089226 -0.80115427]
 [ 0.56561105  0.06541577 -0.6338014   0.52354627]]

Explained variance (%):
[72.77045209 23.03052327  3.68383196  0.51519268]

Cumulative explained variance (%):
[ 72.77045209  95.80